In [2]:
# 练习一下einsum
import torch
from einops import einsum, rearrange

In [3]:
in_features = 10
out_features = 11
x = torch.randn(in_features)
W = torch.randn(out_features, in_features)
x @ W.T, W @ x, einsum(W, x, 'out in, in -> out')

/home/wly/szl_all_code/cs336-all/assignment1/.venv/lib/python3.12/site-packages/einops/parsing.py:137: RuntimeWarning: It is discouraged to use axes names that are keywords: in
  warnings.warn("It is discouraged to use axes names that are keywords: {}".format(name), RuntimeWarning)


(tensor([-5.0685e+00,  3.5538e+00,  2.2830e+00, -1.1364e+00, -9.2223e-01,
          1.9729e-03,  3.3368e+00,  4.0091e-01, -1.4512e+00, -1.7673e+00,
          2.9983e+00]),
 tensor([-5.0685e+00,  3.5538e+00,  2.2830e+00, -1.1364e+00, -9.2223e-01,
          1.9729e-03,  3.3368e+00,  4.0091e-01, -1.4512e+00, -1.7673e+00,
          2.9983e+00]),
 tensor([-5.0685e+00,  3.5538e+00,  2.2830e+00, -1.1364e+00, -9.2223e-01,
          1.9728e-03,  3.3368e+00,  4.0091e-01, -1.4512e+00, -1.7673e+00,
          2.9983e+00]))

In [4]:
batch =6
seq = 5
dim = 4
hid = torch.randn(batch, seq, dim)
A = torch.randn(dim*4, dim)
(hid @ A.T).shape, einsum(hid, A, 'b s d, o d -> b s o').shape

(torch.Size([6, 5, 16]), torch.Size([6, 5, 16]))

In [5]:
hid.sum(dim=-1)

tensor([[-1.7205, -1.5233, -0.5055, -2.5725, -3.3461],
        [ 2.9668,  2.4732, -0.7676,  0.7624, -2.6731],
        [ 2.2599, -1.5451, -2.4231, -1.9515,  0.2565],
        [ 3.0326,  0.0330, -2.7997, -1.8565,  1.0461],
        [ 1.8996, -0.5860,  0.1225, -0.6841,  2.6111],
        [-1.3601, -2.0725,  4.9054,  0.9144,  3.1273]])

In [6]:
a = torch.arange(0,10)
b = torch.arange(0, -12, -1)
c = a.unsqueeze(1) @ b.unsqueeze(0)
c.shape

torch.Size([10, 12])

In [7]:
theta = 10000
d_k = 100
max_seq_len = 200
inv_freqs = theta ** (- torch.arange(0, d_k, 2, dtype=torch.float32) / d_k)
        
# 构建频率和位置
pos = torch.arange(0, max_seq_len, dtype=torch.float32) # (max_seq_len)

# 构建sin和cos
# 我们需要的sin和cos的形状是什么?
# 需要能和 x 进行计算, 那么他们都是2d向量
# freqs = pos.unsqueeze(1) @ inv_freqs.unsqueeze(0)
# 轮椅写法
freqs = einsum(pos, inv_freqs, 's, d -> s d')

cos = torch.cos(freqs) # seq d/2
sin = torch.sin(freqs) # seq d/2
sin, cos

(tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 8.4147e-01,  7.3912e-01,  6.3795e-01,  ...,  1.7378e-04,
           1.4454e-04,  1.2023e-04],
         [ 9.0930e-01,  9.9570e-01,  9.8254e-01,  ...,  3.4756e-04,
           2.8909e-04,  2.4045e-04],
         ...,
         [ 7.9581e-01,  4.7472e-01, -9.3284e-01,  ...,  3.4228e-02,
           2.8471e-02,  2.3682e-02],
         [-7.9579e-02,  9.7029e-01, -9.4820e-01,  ...,  3.4402e-02,
           2.8616e-02,  2.3803e-02],
         [-8.8180e-01,  8.3239e-01, -5.2755e-01,  ...,  3.4575e-02,
           2.8760e-02,  2.3923e-02]]),
 tensor([[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
         [ 0.5403,  0.6736,  0.7701,  ...,  1.0000,  1.0000,  1.0000],
         [-0.4161, -0.0926,  0.1860,  ...,  1.0000,  1.0000,  1.0000],
         ...,
         [-0.6056,  0.8801, -0.3603,  ...,  0.9994,  0.9996,  0.9997],
         [-0.9968,  0.2419,  0.3177,  ...,  0.9994,  0.9

In [8]:
x.max(-1, keepdim=True).values

tensor([1.1760])

In [9]:
torch.tril(torch.ones(4,4, dtype=torch.bool))

tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])

In [10]:
x = torch.randn(10, 11, 12)
x_head = x.reshape(10, 11, 4, 3).permute(0, 2, 1, 3)
y = x_head.transpose(1, 2).reshape(10, 11, 12)
x == y

tensor([[[True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         ...,
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True]],

        [[True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         ...,
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True]],

        [[True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         ...,
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True]],

In [11]:
torch.ones(1,2,dtype=torch.bool)

tensor([[True, True]])

In [12]:
x = torch.randn(2, 10, 3)
x[:, -1, :]

tensor([[ 0.5933, -0.1097, -0.4167],
        [-0.8686, -1.2201,  1.1545]])

In [13]:
x = torch.tensor([[1, 2, 3], [4, 5, 6],[7,8,9]])
row = torch.arange(3)
col = torch.arange(3)
x[row, col]

tensor([1, 5, 9])

In [14]:
import numpy as np
import torch
dataset = np.array([1,2,3,4,5,6,7,8])
batch_size = 3
context_length = 3
start = [torch.randint(0, dataset.shape[-1]-context_length, (batch_size,)) for i in range(batch_size)]
start, torch.stack(start, dim=0)

([tensor([0, 3, 4]), tensor([1, 0, 0]), tensor([2, 0, 3])],
 tensor([[0, 3, 4],
         [1, 0, 0],
         [2, 0, 3]]))

In [19]:
logits = torch.randn(1, 1, 12).squeeze(1)
probs = torch.softmax(logits, dim=-1).squeeze()
torch.multinomial(probs, num_samples=1)

tensor([9])